In [ ]:
from bs4 import BeautifulSoup
from urllib.request import Request, urlopen
import csv
import os
import time
import random
import threading
import queue
from concurrent.futures import ThreadPoolExecutor
import logging
from threading import Lock

# Configure logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("scraper.log"),
        logging.StreamHandler()
    ]
)

# Configuration
MAX_PAGES = 26500
BASE_URL = "https://stackoverflow.com/questions?tab=newest&page="
USER_AGENTS = [
    'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.36',
    'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36',
    'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/90.0.4430.212 Safari/537.36',
    'Mozilla/5.0 (iPhone; CPU iPhone OS 14_0 like Mac OS X) AppleWebKit/537.36 (KHTML, like Gecko) Version/14.0 Mobile/15E148 Safari/604.1'
]
MAX_THREADS = 5  # Adjust based on your connection and target site's limitations
REQUEST_DELAY = (1, 3)  # Random delay between requests (min, max) in seconds
RETRY_DELAY = 5  # Delay in seconds before retrying failed requests
RATE_LIMIT_DELAY = 60  # Delay in seconds when rate limited (429)
MAX_RETRIES = 3  # Maximum number of retries for a failed page

# Thread-safe variables
write_lock = Lock()
question_counter = 0
page_queue = queue.Queue()
completed_pages = set()
error_pages = {}

# Load checkpoint
def load_checkpoint():
    checkpoint_file = 'checkpoint.txt'
    completed_file = 'completed.txt'
    
    start_page = 1
    if os.path.exists(checkpoint_file):
        with open(checkpoint_file, 'r') as f:
            start_page = int(f.read().strip()) + 1
    
    if os.path.exists(completed_file):
        with open(completed_file, 'r') as f:
            for line in f:
                try:
                    completed_pages.add(int(line.strip()))
                except ValueError:
                    pass
    
    return start_page

# Save checkpoint
def save_checkpoint(page_num):
    with open('checkpoint.txt', 'w') as f:
        f.write(str(page_num))
    
    with open('completed.txt', 'a') as f:
        f.write(f"{page_num}\n")

# Worker function for thread pool
def scrape_page(page_num):
    global question_counter
    
    if page_num in completed_pages:
        logging.info(f"Page {page_num} already scraped. Skipping...")
        return True
    
    retries = 0
    while retries < MAX_RETRIES:
        site = f"{BASE_URL}{page_num}"
        hdr = {'User-Agent': random.choice(USER_AGENTS)}
        
        try:
            # Make HTTP request
            req = Request(site, headers=hdr)
            page = urlopen(req)
            soup = BeautifulSoup(page, 'html.parser')
            
            # Find all question elements
            questions = soup.find_all('div', class_='s-post-summary')
            
            if not questions:
                logging.warning(f"No questions found on page {page_num}. Skipping...")
                with write_lock:
                    completed_pages.add(page_num)
                    save_checkpoint(page_num)
                return True
            
            results = []
            with write_lock:
                local_question_counter = question_counter
                for question in questions:
                    local_question_counter += 1
                    
                    # Extract date and time
                    time_element = question.find('span', class_='relativetime')
                    if time_element and 'title' in time_element.attrs:
                        posted_datetime = time_element['title']
                        if 'T' in posted_datetime:
                            date_posted, time_posted = posted_datetime.split('T')
                            time_posted = time_posted.split('+')[0]  # Remove timezone info
                        else:
                            date_posted = posted_datetime
                            time_posted = 'N/A'
                    else:
                        date_posted, time_posted = 'N/A', 'N/A'
                    
                    # Find tags
                    tags = question.find_all('a', class_='post-tag')
                    for tag in tags:
                        tag_text = tag.get_text()
                        results.append([local_question_counter, tag_text, date_posted, time_posted])
                
                # Write results to CSV
                with open(csv_filename, mode='a', newline='', encoding='utf-8') as file:
                    writer = csv.writer(file)
                    writer.writerows(results)
                
                # Update the global counter
                question_counter = local_question_counter
                
                # Mark page as completed
                completed_pages.add(page_num)
                save_checkpoint(page_num)
            
            logging.info(f"✅ Page {page_num} scraped successfully.")
            
            # Add delay to avoid getting blocked
            time.sleep(random.uniform(*REQUEST_DELAY))
            return True
            
        except Exception as e:
            retries += 1
            
            if hasattr(e, 'code') and e.code == 404:
                logging.error(f"❌ Page {page_num} not found (404). Skipping...")
                return False
            elif hasattr(e, 'code') and e.code == 429:
                logging.warning(f"⚠ Too many requests (429) on page {page_num}. Pausing for {RATE_LIMIT_DELAY} seconds...")
                time.sleep(RATE_LIMIT_DELAY)
            else:
                logging.error(f"❌ Error scraping page {page_num}: {e}")
                time.sleep(RETRY_DELAY)
                
            if retries >= MAX_RETRIES:
                with write_lock:
                    error_pages[page_num] = str(e)
                logging.error(f"❌ Failed to scrape page {page_num} after {MAX_RETRIES} attempts.")
                return False
    
    return False

# Main function
def main():
    global csv_filename
    
    start_page = load_checkpoint()
    logging.info(f"Starting scrape from page {start_page} to {MAX_PAGES}")
    
    # Create CSV file
    csv_filename = f'stackoverflow_tags_{start_page}to{MAX_PAGES}.csv'
    csv_exists = os.path.exists(csv_filename)
    
    with open(csv_filename, mode='a' if csv_exists else 'w', newline='', encoding='utf-8') as file:
        writer = csv.writer(file)
        if not csv_exists:
            writer.writerow(['Question', 'Tag', 'Published Date', 'Published Time'])  # CSV header
    
    # Create a queue of pages to process
    for page_num in range(start_page, MAX_PAGES + 1):
        if page_num not in completed_pages:
            page_queue.put(page_num)
    
    # Process pages with thread pool
    with ThreadPoolExecutor(max_workers=MAX_THREADS) as executor:
        while not page_queue.empty():
            # Get a batch of pages to process
            batch = []
            for _ in range(min(MAX_THREADS, page_queue.qsize())):
                if not page_queue.empty():
                    batch.append(page_queue.get())
            
            # Submit batch to thread pool
            futures = [executor.submit(scrape_page, page) for page in batch]
            
            # Wait for all futures to complete
            for future in futures:
                future.result()
    
    # Report on any pages that failed
    if error_pages:
        logging.warning(f"Failed to scrape {len(error_pages)} pages:")
        for page, error in error_pages.items():
            logging.warning(f"  Page {page}: {error}")
    
    logging.info(f"Scraping completed. Data saved to {csv_filename}")

if __name__ == "__main__":
    main()

2025-04-03 00:08:58,969 - INFO - Starting scrape from page 26172 to 26500
--- Logging error ---
Traceback (most recent call last):
  File "c:\Program Files\Python312\Lib\logging\__init__.py", line 1163, in emit
    stream.write(msg + self.terminator)
  File "c:\Program Files\Python312\Lib\encodings\cp1252.py", line 19, in encode
    return codecs.charmap_encode(input,self.errors,encoding_table)[0]
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
UnicodeEncodeError: 'charmap' codec can't encode character '\u2705' in position 33: character maps to <undefined>
Call stack:
  File "c:\Program Files\Python312\Lib\threading.py", line 1032, in _bootstrap
    self._bootstrap_inner()
  File "c:\Program Files\Python312\Lib\threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "C:\Users\Lenovo\AppData\Roaming\Python\Python312\site-packages\ipykernel\ipkernel.py", line 766, in run_closure
    _threading_Thread_run(self)
  File "c:\Program Files\Python312\Lib\threadin

In [1]:
import pandas as pd

# Load CSV file
file_path = "final_merged_stackoverflow_tags.csv"  # Change this to your actual file path
df = pd.read_csv(file_path)

# Convert 'Published Date' to datetime and extract year
df["Published Date"] = pd.to_datetime(df["Published Date"])
df["Year"] = df["Published Date"].dt.year

# Define the target number of tags per year
target_count = 58500

# Trim tags for each year sequentially (keep first 58,500 rows per year)
balanced_df = df.groupby("Year").head(target_count)

# Save the balanced dataset
output_file = "balanced_tags.csv"
balanced_df.to_csv(output_file, index=False)

print(f"Balanced CSV saved as {output_file}")


Balanced CSV saved as balanced_tags.csv


In [3]:
import pandas as pd

# Load CSV file
file_path = "stackoverflow_tags.csv"  # Change this to your actual file path
df = pd.read_csv(file_path)

# Convert 'Published Date' to datetime and extract year
df["Published Date"] = pd.to_datetime(df["Published Date"])
df["Year"] = df["Published Date"].dt.year

# Define the target number of tags per year
target_count = 58500

# Trim tags for each year sequentially (keep first 58,500 rows per year)
df_2025 = df[df["Year"] == 2025].head(target_count)
df_2024 = df[df["Year"] == 2024].head(target_count)
df_2023 = df[df["Year"] == 2023].head(target_count)

# Concatenate in the required order (2025 → 2024 → 2023)
balanced_df = pd.concat([df_2025, df_2024, df_2023])

# Save the balanced dataset
output_file = "balanced_tag.csv"
balanced_df.to_csv(output_file, index=False)

print(f"Balanced CSV saved as {output_file}")


Balanced CSV saved as balanced_tag.csv
